# <b>The Skeleton Run

---

---

# 1. The Setup

## 1. Importing Libraries

In [1]:
import os
import torch
import torch.nn as nn

import pymupdf
import re
import fitz
import pandas as pd

from transformers import(
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    DistilBertModel
)

from peft import PeftModel

## 2. Device Configuration

In [2]:
device = "mps"

## 3. Locate the models.

In [3]:
BASE_DIR = "../models/"

### 3.1 Locate the text-classifier

In [4]:
MIRA_DISTILBERT_LORA = f"{BASE_DIR}/mira-distilbert-lora"

### 3.2 Locate the text-classifier heads.

In [5]:
MIRA_CLASSIFIER_HEADS = f"{BASE_DIR}/mira-distilbert-lora"

### 3.3 Locate the Risk Engine.

In [6]:
MIRA_RISK_CLASSIFIER = f"{BASE_DIR}/mira_risk_engine/mira-risk-classifier.pkl"

In [7]:
MIRA_RISK_ENCODER = f"{BASE_DIR}/mira_risk_engine/mira-risk-encoder.pkl"

### 3.3 Locate the GenLLM.

In [8]:
GEN_MODEL_PATH = f"{BASE_DIR}/MIRA2_qwen2.5-3b-lora"

---

## 4. Load <b>Inspection Classifier</b> for PDF's text.

In [9]:
class MIRAClassifier(nn.Module):
    def __init__(self, num_issue, num_category, num_severity, num_recurring):
        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')

        hidden_size = self.distilbert.config.hidden_size

        # sub-models
        self.issue_classifier = nn.Linear(hidden_size, num_issue)
        self.category_classifier = nn.Linear(hidden_size, num_category)
        self.severity_classifier = nn.Linear(hidden_size, num_severity)
        self.recurring_classifier = nn.Linear(hidden_size, num_recurring)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        issue_logits = self.issue_classifier(pooled_output)
        category_logits = self.category_classifier(pooled_output)
        severity_logits = self.severity_classifier(pooled_output)
        recurring_logits = self.recurring_classifier(pooled_output)

        return {
            'issue': issue_logits,
            'category': category_logits,
            'severity': severity_logits,
            'recurring': recurring_logits
        }

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [11]:
num_issue = 10
num_category = 7
num_severity = 4
num_recurring = 2

In [12]:
inspection_classifier = MIRAClassifier(
    num_issue = num_issue,
    num_category = num_category,
    num_severity = num_severity,
    num_recurring = num_recurring
)

In [13]:
inspection_classifier.distilbert = PeftModel.from_pretrained(
    inspection_classifier.distilbert,
    MIRA_DISTILBERT_LORA
)

In [14]:
inspection_classifier = inspection_classifier.to(device)

### 4.1 Load Classification Heads

In [15]:
heads = torch.load(
    os.path.join(
        MIRA_CLASSIFIER_HEADS,
        "classification_heads.pth"
    ),
    map_location=device
)

inspection_classifier.issue_classifier.load_state_dict(
    heads["issue_classifier"]
)

inspection_classifier.category_classifier.load_state_dict(
    heads["category_classifier"]
)

inspection_classifier.severity_classifier.load_state_dict(
    heads["severity_classifier"]
)

inspection_classifier.recurring_classifier.load_state_dict(
    heads["recurring_classifier"]
)

inspection_classifier = inspection_classifier.to(device)
inspection_classifier.eval()

MIRAClassifier(
  (distilbert): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): MultiHeadSelfAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=

### 4.2 Load Label Mappings

In [16]:
import json

with open(
    os.path.join(
        MIRA_DISTILBERT_LORA,
        'label_mappings.json'
    ),
    'r'
) as f:
    label_mappings = json.load(f)

---

## 5. Load <b>Risk Engine

In [17]:
import joblib

risk_model = joblib.load(MIRA_RISK_CLASSIFIER)
risk_encoder = joblib.load(MIRA_RISK_ENCODER)

/Users/tanmaykhomane/Artificial Intelligence/GenAI IBM/llm_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/tanmaykhomane/Artificial Intelligence/GenAI IBM/llm_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---

## 6. Load <b>GenLLM</b>

In [18]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LORA_PATH = GEN_MODEL_PATH

In [19]:
genLLM_tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

In [20]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16
).to(device)

genLLM_model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

genLLM_model = genLLM_model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

---

---

# HELPERs

---

### PDF Processing

In [21]:
PDF_PATH = "AI/Knowledge Bases/Coal_Mine_Inspection_Report_Concise.pdf"

Extract Text from PDF

In [22]:
def extract_pdf_text(pdf_path):

    document = fitz.open(
        pdf_path
    )

    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [23]:
def extract_findings(text):

    pattern = r'Finding\s+(F-\d+):\s*(.*?)(?=Finding\s+F-\d+:|(?:\n|\s)5\.\s*INSPECTOR[\'’]?S\s+REMARKS|$)'

    matches = re.findall(
        pattern,
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    findings = []

    for finding_id, finding_text in matches:

        findings.append({
            'finding_id': finding_id,
            'finding_text': finding_text.strip()
        })

    return findings

### Findings' Classification

In [24]:
def classify_finding(text):
    inputs = tokenizer(
        text, 
        return_tensors = 'pt',
        truncation = True,
        padding = True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = inspection_classifier(
            input_ids = inputs['input_ids'],
            attention_mask = inputs['attention_mask']
        )

    issue_id = torch.argmax(
        outputs['issue'],
        dim=1
    ).item()

    category_id = torch.argmax(
        outputs['category'],
        dim=1
    ).item()

    severity_id = torch.argmax(
        outputs['severity'],
        dim=1
    ).item()

    recurring_id = torch.argmax(
        outputs['recurring'],
        dim=1
    ).item()

    return {
        'issue': label_mappings['issue'][issue_id],
        'category': label_mappings['category'][category_id],
        'severity': label_mappings['severity'][severity_id],
        'recurring': label_mappings['recurring'][recurring_id]
    }

In [25]:
def classify_findings(findings):
    results = []

    for finding in findings:
        prediction = classify_finding(finding['finding_text'])

        results.append({
                'finding_id': finding['finding_id'],
                'finding_text': finding['finding_text'],
                'issue': prediction['issue'],
                'category': prediction['category'],
                'severity': prediction['severity'],
                'recurring': prediction['recurring']
            })

    return results

### Predict Risk for all findings

In [26]:
def predict_risks(results):

    risk_results = []

    for result in results:

        risk_input = pd.DataFrame([{
            "issue": result["issue"],
            "category": result["category"],
            "severity": result["severity"],
            "recurring": result["recurring"]
        }])

        risk_input_encoded = risk_encoder.transform(
            risk_input
        )

        predicted_risk = risk_model.predict(
            risk_input_encoded
        )[0]

        probabilities = risk_model.predict_proba(
            risk_input_encoded
        )[0]

        predicted_index = list(
            risk_model.classes_
        ).index(predicted_risk)

        risk_confidence = (
            probabilities[predicted_index] * 100
        )

        risk_results.append({
            "finding_id": result["finding_id"],
            "finding_text": result["finding_text"],
            "issue": result["issue"],
            "category": result["category"],
            "severity": result["severity"],
            "recurring": result["recurring"],
            "risk_level": predicted_risk,
            "risk_confidence": round(
                risk_confidence,
                2
            )
        })

    return risk_results

### Text Generator Function

In [27]:
from transformers import TextIteratorStreamer
from threading import Thread

In [28]:
def generate_response(user_prompt, max_new_tokens=300):

    genLLM_model.eval()

    messages = [
        {
            "role": "system",
            "content": (
                "You are MIRA (Mine Intelligence and Risk Assessment), "
                "an AI-based coal mine inspection and compliance assistant. "

                "Use only the supplied inspection findings, structured AI "
                "assessment, risk information, and retrieved regulatory guidance "
                "to answer the user's question. "

                "Do not calculate, modify, or override the provided issue, "
                "category, severity, recurring status, risk level, or risk confidence. "

                "Do not invent regulations, rule numbers, legal provisions, "
                "inspection history, or evidence that is not present in the context. "

                "For compliance-related questions, use only the Retrieved Guidance "
                "provided in the user prompt. "

                "If Language is English, respond in clear professional English. "

                "If Language is Hindi, understand Romanized Hindi or Hinglish "
                "input and respond ONLY in standard Hindi using Devanagari script. "
                "Do not respond in Romanized Hindi or English."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = genLLM_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = genLLM_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(genLLM_model.device)

    streamer = TextIteratorStreamer(
        genLLM_tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": genLLM_tokenizer.pad_token_id,
        "eos_token_id": genLLM_tokenizer.eos_token_id
    }

    thread = Thread(
        target = genLLM_model.generate,
        kwargs = generation_kwargs
    )

    thread.start()

    response = ""

    for new_text in streamer:
        print(new_text, end="", flush=True)
        response += new_text

    thread.join()

### Prompt Builder 

In [29]:
def build_genllm_prompt(
    user_query,
    language,
    risk_results,
    retrieved_guidance
):

    findings_context = ""

    for result in risk_results:

        findings_context += f"""
Finding ID: {result['finding_id']}

Finding:

{result['finding_text']}

Issue: {result['issue']}
Category: {result['category']}
Severity: {result['severity']}
Recurring: {result['recurring']}
Risk Level: {result['risk_level']}
Risk Confidence: {result['risk_confidence']}%

"""

    return f"""Language: {language}

User Query: {user_query}

Inspection Findings:

{findings_context}

Retrieved Guidance:

{retrieved_guidance}
"""

---

# RAG

In [30]:
import faiss
from pypdf import PdfReader

from transformers import (DPRContextEncoder, DPRContextEncoderTokenizer,
                          DPRQuestionEncoder, DPRQuestionEncoderTokenizer)

In [31]:
KNOWLEDGE_BASE = "../Knowledge Bases/Coal_Mines_Regulation_2017_Noti.pdf"

In [32]:
reader = PdfReader(KNOWLEDGE_BASE)
print("Number of pages: ", len(reader.pages))

Number of pages:  280


In [33]:
def read_and_split_pdf(filename):
    reader = PdfReader(filename)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + '\n'

    paragraphs = text.split("\n")

    return [
        para.strip()
        for para in paragraphs
        if len(para.strip()) > 0
    ]

In [34]:
paragraphs = read_and_split_pdf(KNOWLEDGE_BASE)
print("Number of paragraphs: ", len(paragraphs))

Number of paragraphs:  11726


In [38]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_encoder = context_encoder.to(device)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mod

In [42]:
def encode_contexts(text_list, batch_size=8):
    embeddings = []
    context_encoder.eval()

    for i in range(0, len(text_list), batch_size):

        batch = text_list[i:i + batch_size]

        inputs = context_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            outputs = context_encoder(
                **inputs
            )

        embeddings.append(
            outputs.pooler_output.cpu()
        )

    return torch.cat(embeddings).numpy()

In [40]:
context_embeddings = encode_contexts(paragraphs)

In [43]:
MIRA_RAG_PATH = "../Knowledge Bases/Encodings/"

In [45]:
import faiss

embedding_dimension = context_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dimension)
faiss_index.add(context_embeddings)

In [46]:
MIRA_FAISS_INDEX = os.path.join(
    MIRA_RAG_PATH,
    "regulatory.index"
)

In [ ]:
faiss.write_index(faiss_index, MIRA_FAISS_INDEX)

In [49]:
import pickle

MIRA_RAG_CHUNKS = os.path.join(
    MIRA_RAG_PATH,
    "regulatory_chunks.pkl"
)

with open(
    MIRA_RAG_CHUNKS,
    "wb"
) as f:
    pickle.dump(paragraphs, f)

----

----

# The SKELETON

In [ ]:
pdf_path = "../Knowledge Bases/Coal_Mine_Inspection_Report_Concise.pdf"

# 1. Extract text from PDF
pdf_text = extract_pdf_text(
    pdf_path
)

# 2. Extract findings
findings = extract_findings(
    pdf_text
)

print(
    "Number of findings:",
    len(findings)
)

# 3. Classify all findings
results = classify_findings(
    findings
)

# 4. Predict risk for every finding
risk_results = predict_risks(
    results
)

Number of findings: 6
